# Learning When Not to Use a Battery — Extension Paper

**Author:** Team Dynamic  
**Based on:** Shikdar & Laaksonen (2026) — *International Transactions on Electrical Energy Systems*  
**Description:** This notebook reproduces and extends the original paper with 7 improvements:
1. Continuous service horizons (L1)
2. Multi-model benchmarking (L2)
3. Composite failure definition (L3)
4. Cross-chemistry validation (L4)
5. Field-like data augmentation (L5)
6. Market simulation (L6)
7. Continuous derating dispatch (L7)

---
## 0. Environment Setup

In [ ]:
# Install dependencies
!pip install -q numpy pandas scipy scikit-learn matplotlib seaborn xgboost tensorflow tqdm pyyaml python-docx

# Check GPU
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")
if tf.config.list_physical_devices('GPU'):
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive to persist results across sessions
from google.colab import drive
drive.mount('/content/drive')

# Set up working directory
import os
WORK_DIR = '/content/drive/MyDrive/battery_extension'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")

In [ ]:
# Verify code modules are accessible
if not os.path.exists('code/src') and not os.path.exists('src'):
    print("Please upload the code/ directory to your Drive at:")
    print(f"  {WORK_DIR}/code/")
    print("Then re-run this cell.")
else:
    print("Code modules found. Ready for imports.")

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, yaml, os, sys, warnings
warnings.filterwarnings('ignore')

# Auto-detect the src/ directory location
_FOUND = False
_CANDIDATES = [
    os.path.abspath('code'),
    os.path.abspath('.'),
    '/content/drive/MyDrive/battery_extension/code',
    '/content/drive/MyDrive/battery_extension',
]
# Also check immediate parent (in case notebook is inside code/)
_PARENT = os.path.dirname(os.path.abspath('.'))
if os.path.isdir(os.path.join(_PARENT, 'src')):
    _CANDIDATES.insert(0, _PARENT)
for _p in _CANDIDATES:
    if os.path.isdir(_p) and os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p)
        print(f"src/ found at: {os.path.join(_p, 'src')}")
        _FOUND = True
        break
if not _FOUND:
    raise ImportError(
        "Cannot find src/ directory. Ensure the 'code' folder is uploaded "
        "to /content/drive/MyDrive/battery_extension/ and re-run the mount cell.")

from src.data.nasa import NASALoader
from src.data.calce import CALCELoader
from src.data.composite_failure import CompositeFailureLabeler
from src.data.augmentation import OperationalAugmenter
from src.models.xgboost_hazard import XGBoostHazard
from src.models.lstm_hazard import LSTMHazard
from src.models.tcn_hazard import TCNHazard
from src.models.transformer_hazard import TransformerHazard
from src.models.continuous_hazard import ContinuousHazard
from src.models.calibration import ProbabilityCalibrator
from src.dispatch.threshold import ThresholdPolicy
from src.dispatch.derating import ContinuousDeratingPolicy
from src.dispatch.rl_policy import RLThresholdPolicy
from src.dispatch.market_sim import MarketSimulator
from src.evaluation.metrics import compute_metrics, compute_operational_metrics
from src.evaluation.visualization import (
    plot_calibration, plot_risk_tradeoff, plot_survival,
    plot_multi_horizon, plot_auc_comparison, save_figure)

print("All imports successful.")

In [ ]:
with open('code/config.yaml') as f:
    cfg = yaml.safe_load(f)

print("Configuration loaded.")
horizons = cfg['horizons']
horizon_cols = [f'fail_{h}' for h in horizons]
feature_cols = cfg['features']['input_cols']
print(f"Horizons: {horizons}")
print(f"Features: {feature_cols}")

---
## 1. Data Pipeline

Load NASA classic dataset (same as original paper) and CALCE datasets (different chemistries).

In [ ]:
print("Loading NASA classic dataset...")
loader = NASALoader(data_dir='data')
df_nasa = loader.load_classic()
print(f"NASA classic: {len(df_nasa)} rows, {df_nasa['cell_id'].nunique()} cells")
print(f"Cells: {df_nasa['cell_id'].unique()}")
df_nasa.head(3)

In [ ]:
print("Loading CALCE datasets...")
calce = CALCELoader(data_dir='data', chemistries=['LCO', 'LFP', 'K2'])
df_calce = calce.load_all()
print(f"CALCE: {len(df_calce)} rows, {df_calce['cell_id'].nunique()} cells")
if not df_calce.empty:
    print(f"Chemistries: {df_calce['chemistry'].unique() if 'chemistry' in df_calce.columns else 'N/A'}")
    df_calce.head(3)

---
## 2. Failure Labels

Two labeling strategies:
- **Single** (original): SOH < 70%
- **Multi** (extension): SOH + sudden capacity drop + resistance rise

In [ ]:
labeler = CompositeFailureLabeler(
    soh_threshold=cfg['failure']['soh_threshold'],
    sudden_drop=cfg['failure']['sudden_drop_threshold'])

# Single-criterion (original)
df_single = labeler.label(df_nasa.copy(), method='single')

# Multi-criteria (extension)
df_multi = labeler.label(df_nasa.copy(), method='multi')

print("Failure event counts (single):")
for h in horizon_cols:
    print(f"  {h}: {df_single[h].sum()}")
print("\nFailure event counts (multi):")
for h in horizon_cols:
    print(f"  {h}: {df_multi[h].sum()}")

In [ ]:
print("Augmenting data with operational variability...")
augmenter = OperationalAugmenter(seed=42)
df_aug = augmenter.augment(df_single, n_virtual_cells=3)
print(f"After augmentation: {len(df_aug)} rows, {df_aug['cell_id'].nunique()} cells")

---
## 3. Experiment 1: Baseline Replication

Reproduce the original paper: XGBoost + single label + leave-battery-out CV.

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: Baseline Replication")
print("=" * 60)

window = cfg['features']['window_size']
xg_cfg = cfg['models']['xgboost']
xg_cfg['window_size'] = window

calibrator = ProbabilityCalibrator(method=cfg['calibration']['method'])

results_baseline = leave_battery_out_cv(
    df_single, XGBoostHazard, xg_cfg, feature_cols, horizon_cols, calibrator)

metrics_raw = compute_metrics(results_baseline['targets'], results_baseline['predictions'],
                                horizons=horizons)
metrics_cal = compute_metrics(results_baseline['targets'], results_baseline['calibrated'],
                                horizons=horizons)

print("\n--- Raw ---")
for h in horizons:
    m = metrics_raw['per_horizon'][h]
    print(f"  H={h}: AUC={m['auc']}, Brier={m['brier']}, ECE={m['ece']}")
print(f"  Macro avg: AUC={metrics_raw['macro_avg']['auc']}")

print("\n--- Calibrated ---")
for h in horizons:
    m = metrics_cal['per_horizon'][h]
    print(f"  H={h}: AUC={m['auc']}, Brier={m['brier']}, ECE={m['ece']}")
print(f"  Macro avg: AUC={metrics_cal['macro_avg']['auc']}")

---
## 4. Experiment 2: Multi-Model Benchmark (L2)

Compare XGBoost, LSTM, TCN, and Transformer on the same data.

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: Multi-Model Benchmark")
print("=" * 60)

model_registry = {
    'XGBoost': XGBoostHazard,
    'LSTM': LSTMHazard,
    'TCN': TCNHazard,
    'Transformer': TransformerHazard,
}

all_model_scores = {}

for name, cls in model_registry.items():
    print(f"\n>>> {name}")
    mcfg = cfg['models'].get(name.lower(), cfg['models']['xgboost'])
    mcfg['window_size'] = window
    try:
        cv = leave_battery_out_cv(
            df_single, cls, mcfg, feature_cols, horizon_cols,
            calibrator if name == 'XGBoost' else None)
        m = compute_metrics(cv['targets'], cv['predictions'], horizons=horizons)
        all_model_scores[name] = m['macro_avg']
        print(f"  AUC={m['macro_avg']['auc']:.4f}, Brier={m['macro_avg']['brier']:.4f}")
    except Exception as e:
        print(f"  FAILED: {e}")

print("\n--- Model Summary ---")
for name, s in all_model_scores.items():
    print(f"  {name:15s}: AUC={s['auc']:.4f}")

In [ ]:
# AUC comparison plot
fig, ax = plt.subplots(figsize=(6, 4))
plot_auc_comparison(all_model_scores, ax)
plt.show()

---
## 5. Experiment 3: Cross-Chemistry Validation (L4)

Train on NASA (LCO/LiCoO2), evaluate on CALCE LFP and K2 chemistries.

In [ ]:
print("=" * 60)
print("EXPERIMENT 3: Cross-Chemistry Validation")
print("=" * 60)

chemistry_results = {}

for chem in ['LCO', 'LFP', 'K2']:
    print(f"\nTesting on {chem}...")
    calce = CALCELoader(data_dir='data', chemistries=[chem])
    try:
        df_test = calce.load_all()
    except Exception as e:
        print(f"  Error: {e}")
        continue
    if df_test.empty:
        print(f"  No data for {chem}, skipping.")
        continue
    df_test = labeler.label(df_test, method='single')
    combined = pd.concat([df_single, df_test], ignore_index=True)
    cv = leave_battery_out_cv(
        combined, XGBoostHazard, xg_cfg, feature_cols, horizon_cols, calibrator)
    m = compute_metrics(cv['targets'], cv['calibrated'], horizons=horizons)
    chemistry_results[chem] = m['macro_avg']
    print(f"  AUC={m['macro_avg']['auc']:.4f}")

print("\n--- Cross-Chemistry Summary ---")
for chem, s in chemistry_results.items():
    print(f"  {chem:10s}: AUC={s['auc']:.4f}")

---
## 6. Experiment 4: Composite Failure Label (L3)

In [ ]:
print("=" * 60)
print("EXPERIMENT 4: Composite vs Single Failure Labels")
print("=" * 60)

for label_name, df_labeled in [('Single (SOH<70%)', df_single),
                                 ('Multi (Composite)', df_multi)]:
    print(f"\n{label_name}")
    cv = leave_battery_out_cv(
        df_labeled, XGBoostHazard, xg_cfg, feature_cols, horizon_cols, calibrator)
    m = compute_metrics(cv['targets'], cv['calibrated'], horizons=horizons)
    print(f"  AUC={m['macro_avg']['auc']:.4f}, Brier={m['macro_avg']['brier']:.4f}")
    
    # Count label balance
    pos_ratio = df_labeled[horizon_cols].values.mean()
    print(f"  Positive label ratio: {pos_ratio:.4f}")

---
## 7. Experiment 5: Dispatch Policy Comparison (L1 + L7)

In [ ]:
print("=" * 60)
print("EXPERIMENT 5: Dispatch Policy Comparison")
print("=" * 60)

P_cal = results_baseline['calibrated']
y_true = results_baseline['targets']

policies = {
    'Always dispatch': lambda p, e: (e * np.ones_like(p), np.ones_like(p, dtype=bool)),
    'Threshold(τ=0.2)': ThresholdPolicy(tau=0.20),
    'Threshold(τ=0.1)': ThresholdPolicy(tau=0.10),
    'Derating(α=2)': ContinuousDeratingPolicy(alpha=2.0),
    'Derating(α=5)': ContinuousDeratingPolicy(alpha=5.0),
}

dispatch_results = {}
for pname, policy in policies.items():
    E_offered, _ = policy.decide(P_cal[:, 0], 0.5)
    n_dispatch = (E_offered > 0).sum()
    total_energy = E_offered.sum()
    failures = (y_true[:, 0] * (E_offered > 0)).sum()
    failure_rate = failures / n_dispatch if n_dispatch > 0 else 0
    dispatch_results[pname] = {
        'energy': round(total_energy, 2),
        'failure_rate': round(failure_rate, 4),
        'n_dispatch': n_dispatch
    }
    print(f"  {pname:25s}: energy={total_energy:.2f}, fail_rate={failure_rate:.4f}")

print("\n--- Dispatch Summary ---")
print(pd.DataFrame(dispatch_results).T)

---
## 8. Experiment 6: Continuous Horizon Hazard (L1)

Train a model that accepts horizon H as an input, enabling prediction at *any* horizon.

In [ ]:
print("=" * 60)
print("EXPERIMENT 6: Continuous Horizon Hazard")
print("=" * 60)

# Flatten data: create (features, horizon) pairs
X_list, H_list, y_list = [], [], []
window = cfg['features']['window_size']

for cell_id, group in df_single.groupby('cell_id'):
    group = group.sort_values('cycle').reset_index(drop=True)
    for t in range(window, len(group)):
        features = group[feature_cols].iloc[t-window:t].values[-1]  # last timestep
        for h in horizons:
            X_list.append(features)
            H_list.append(h)
            y_list.append(group[f'fail_{h}'].iloc[t-1])  # previous state

X_arr = np.array(X_list, dtype=np.float32)
H_arr = np.array(H_list, dtype=np.float32).reshape(-1, 1)
y_arr = np.array(y_list, dtype=np.float32)

print(f"Training data: {len(X_arr)} samples, {X_arr.shape[1]} features")

# Train/val split
split = int(len(X_arr) * 0.8)
idx = np.random.RandomState(42).permutation(len(X_arr))
train_idx, val_idx = idx[:split], idx[split:]

# Normalise features
mu, sigma = X_arr[train_idx].mean(axis=0), X_arr[train_idx].std(axis=0)
X_arr = (X_arr - mu) / (sigma + 1e-8)

cont_model = ContinuousHazard(feature_dim=X_arr.shape[1], config=cfg['models']['lstm'])
cont_model.fit(
    X_arr[train_idx], H_arr[train_idx], y_arr[train_idx],
    X_arr[val_idx], H_arr[val_idx], y_arr[val_idx])

# Evaluate at arbitrary horizons
print("\nEvaluating at standard horizons...")
for h in [10, 15, 20, 25, 30, 40, 50, 60]:
    h_arr = np.full((len(X_arr[val_idx]), 1), h, dtype=np.float32)
    preds = cont_model.predict_proba(X_arr[val_idx], h_arr)
    # Binarize at 0.5 for accuracy
    acc = ((preds > 0.5) == y_arr[val_idx]).mean()
    print(f"  H={h:3d}: accuracy={acc:.4f}")

---
## 9. Experiment 7: Market Simulation (L6)

Evaluate dispatch policies under stochastic prices with revenue tracking.

In [ ]:
print("=" * 60)
print("EXPERIMENT 7: Market Simulation")
print("=" * 60)

# Find eol_cycle from the test battery
eol_cycle = df_single['eol_cycle'].dropna().iloc[0] if df_single['eol_cycle'].notna().any() else 200
P_sample = P_cal[:150, 0]  # Use first 150 cycles

ms = MarketSimulator(
    price_mean=cfg['market']['price_mean'],
    price_std=cfg['market']['price_std'],
    price_ar_coeff=cfg['market']['price_ar_coeff'],
    service_energy_kwh=cfg['market']['service_energy_kwh'],
    penalty_cost=cfg['market']['penalty_cost'],
    seed=42)

market_results = {}
for pname, policy in policies.items():
    if pname == 'Always dispatch':
        policy_obj = lambda p, e: (e * np.ones_like(p), np.ones_like(p, dtype=bool))
    else:
        policy_obj = policy
    mc = ms.monte_carlo(
        P_sample, eol_cycle, horizon=20,
        dispatch_policy=policy_obj if hasattr(policy, 'decide') else None,
        n_scenarios=min(cfg['market']['n_scenarios'], 200))
    market_results[pname] = {
        'mean_revenue': round(mc['revenue'].mean(), 2),
        'std_revenue': round(mc['revenue'].std(), 2),
        'mean_energy': round(mc['energy_delivered'].mean(), 2),
        'mean_failure_rate': round(mc['failure_rate'].mean(), 4),
    }
    print(f"  {pname:25s}: revenue={market_results[pname]['mean_revenue']:.2f} ± "
          f"{market_results[pname]['std_revenue']:.2f}")

---
## 10. Results Summary

All results are collected below in a single table.

In [ ]:
print("=" * 80)
print("RESULTS SUMMARY — Extension Paper")
print("=" * 80)

summary = []

# Baseline
summary.append({'Experiment': 'Baseline', 'Metric': 'AUC (raw, H=20)',
                'Value': metrics_raw['per_horizon'][20]['auc']})
summary.append({'Experiment': 'Baseline', 'Metric': 'AUC (cal, H=20)',
                'Value': metrics_cal['per_horizon'][20]['auc']})
summary.append({'Experiment': 'Baseline', 'Metric': 'Brier (cal, H=20)',
                'Value': metrics_cal['per_horizon'][20]['brier']})

# Models
for name, s in all_model_scores.items():
    summary.append({'Experiment': f'Model: {name}', 'Metric': 'AUC',
                    'Value': s['auc']})

# Chemistry
for chem, s in chemistry_results.items():
    summary.append({'Experiment': f'Chemistry: {chem}', 'Metric': 'AUC',
                    'Value': s['auc']})

# Dispatch
for name, d in dispatch_results.items():
    summary.append({'Experiment': f'Dispatch: {name}', 'Metric': 'Energy (kWh)',
                    'Value': d['energy']})
    summary.append({'Experiment': f'Dispatch: {name}', 'Metric': 'Failure Rate',
                    'Value': d['failure_rate']})

# Market
for name, d in market_results.items():
    summary.append({'Experiment': f'Market: {name}', 'Metric': 'Mean Revenue',
                    'Value': d['mean_revenue']})

results_df = pd.DataFrame(summary)
display(results_df)

# Save
results_df.to_csv('results_summary.csv', index=False)
print("\nResults saved to results_summary.csv")

---
## 11. Generate Figures for the Paper

In [ ]:
os.makedirs('figures', exist_ok=True)

# Fig 1: Calibration curve
fig, ax = plt.subplots(figsize=(6, 5))
plot_calibration(results_baseline['targets'][:, 0],
                 results_baseline['calibrated'][:, 0], ax=ax)
plt.tight_layout()
plt.savefig('figures/calibration_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figures/calibration_curve.png")

# Fig 2: Model comparison
fig, ax = plt.subplots(figsize=(6, 4))
plot_auc_comparison(all_model_scores, ax)
plt.tight_layout()
plt.savefig('figures/model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figures/model_comparison.png")

# Fig 3: Dispatch trade-off
plot_data = {}
for tau in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    p = ThresholdPolicy(tau=tau)
    E, _ = p.decide(P_cal[:, 0], 0.5)
    n_d = (E > 0).sum()
    fail = (y_true[:, 0] * (E > 0)).sum()
    plot_data[f'τ={tau}'] = {
        'failure_rate': fail / n_d if n_d > 0 else 0,
        'energy': E.sum()}
fig, ax = plt.subplots(figsize=(7, 5))
plot_risk_tradeoff(plot_data, ax)
plt.tight_layout()
plt.savefig('figures/dispatch_tradeoff.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figures/dispatch_tradeoff.png")

---
## 12. Save All Results

Package results for paper generation.

In [ ]:
import json

all_results = {
    'baseline_raw': metrics_raw,
    'baseline_calibrated': metrics_cal,
    'model_comparison': {k: v for k, v in all_model_scores.items()},
    'cross_chemistry': chemistry_results,
    'dispatch': dispatch_results,
    'market': market_results,
}

with open('all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("All results saved to all_results.json")
print("\nNext steps:")
print("  1. Download the figures/ directory")
print("  2. Run the paper generation script to produce the manuscript")